# 90819-Python Final Project: Parcel Data Pull

## Step 1: Set-up
In this step, we pull our parcel and municipalities data and shapefiles. The parcel level data is far too large keep in Github, so we saved the data to a google drive. This document is just to review our process of setting up, cleaning, and exporting the data.

In order we will:
1. Install necessary libraries 
2. Load necessary packages
3. read in and describe data

In [20]:
# install libraries
# %pip install seaborn
# %pip install geopandas
# %pip install census
# %pip install pandas
# %pip install us
# %pip install pytidycensus

In [21]:
# import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import pytidycensus as tc
from dotenv import load_dotenv
import os
from pathlib import Path


In [22]:
# READ IN DATA
# Allegheny County property assessments dataframe
assessments_df = pd.read_csv("property_assessments.csv")
# Allegheny County municipalities dataframe
municipalities_df = pd.read_csv("allegheny_municipalities.csv")
# Allegheny County parcel boundaries
parcel_boundaries_df = pd.read_csv("allegheny_parcel_boundaries.csv")
# Allegheny County municipal boundaries geopandas dataframe
municipal_boundaries_gdf = gpd.read_file("allegheny_municipal_boundaries.geojson")

/var/folders/92/rrvsc6_x7slf490khsx_xn2c0000gn/T/ipykernel_69580/3229317460.py:3: DtypeWarning: Columns (19,20,29,30,31,37,38,57,83) have mixed types. Specify dtype option on import or set low_memory=False.
  assessments_df = pd.read_csv("property_assessments.csv")


In [ ]:
# View the data

print(assessments_df.shape)
display(assessments_df.head())

print(parcel_boundaries_df.shape)
display(parcel_boundaries_df.head())

print(municipal_boundaries_gdf.shape)
display(municipal_boundaries_gdf.head())

['PARID', 'PROPERTYHOUSENUM', 'PROPERTYFRACTION', 'PROPERTYADDRESS', 'PROPERTYCITY', 'PROPERTYSTATE', 'PROPERTYUNIT', 'PROPERTYZIP', 'MUNICODE', 'MUNIDESC', 'SCHOOLCODE', 'SCHOOLDESC', 'LEGAL1', 'LEGAL2', 'LEGAL3', 'NEIGHCODE', 'NEIGHDESC', 'TAXCODE', 'TAXDESC', 'TAXSUBCODE', 'TAXSUBCODE_DESC', 'OWNERCODE', 'OWNERDESC', 'CLASS', 'CLASSDESC', 'USECODE', 'USEDESC', 'LOTAREA', 'HOMESTEADFLAG', 'FARMSTEADFLAG', 'CLEANGREEN', 'ABATEMENTFLAG', 'RECORDDATE', 'SALEDATE', 'SALEPRICE', 'SALECODE', 'SALEDESC', 'DEEDBOOK', 'DEEDPAGE', 'PREVSALEDATE', 'PREVSALEPRICE', 'PREVSALEDATE2', 'PREVSALEPRICE2', 'CHANGENOTICEADDRESS1', 'CHANGENOTICEADDRESS2', 'CHANGENOTICEADDRESS3', 'CHANGENOTICEADDRESS4', 'COUNTYBUILDING', 'COUNTYLAND', 'COUNTYTOTAL', 'COUNTYEXEMPTBLDG', 'LOCALBUILDING', 'LOCALLAND', 'LOCALTOTAL', 'FAIRMARKETBUILDING', 'FAIRMARKETLAND', 'FAIRMARKETTOTAL', 'STYLE', 'STYLEDESC', 'STORIES', 'YEARBLT', 'EXTERIORFINISH', 'EXTFINISH_DESC', 'ROOF', 'ROOFDESC', 'BASEMENT', 'BASEMENTDESC', 'GRADE', 

,PARID,PROPERTYHOUSENUM,PROPERTYFRACTION,PROPERTYADDRESS,PROPERTYCITY,PROPERTYSTATE,PROPERTYUNIT,PROPERTYZIP,MUNICODE,MUNIDESC,...,HALFBATHS,HEATINGCOOLING,HEATINGCOOLINGDESC,FIREPLACES,BSMTGARAGE,FINISHEDLIVINGAREA,CARDNUMBER,ALT_ID,TAXYEAR,ASOFDATE
0,0001C00228000400,424.0,,STANWIX ST,PITTSBURGH,PA,,15222.0,101,1st Ward - PITTSBURGH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,30-SEP-26
1,0001D00115000000,500.0,,LIBERTY AVE,PITTSBURGH,PA,,15222.0,101,1st Ward - PITTSBURGH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,30-SEP-26
2,0001D00117000000,502.0,-510,LIBERTY AVE,PITTSBURGH,PA,,15222.0,101,1st Ward - PITTSBURGH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,30-SEP-26
3,0001D00125000000,100.0,,5TH AVE,PITTSBURGH,PA,,15222.0,101,1st Ward - PITTSBURGH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,30-SEP-26
4,0001D00127000000,23.0,,MARKET SQ,PITTSBURGH,PA,,15222.0,101,1st Ward - PITTSBURGH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,30-SEP-26


The property assessments data contains 585,005 rows and 86 columns. The primary columns that we need are the: 
- PARID (parcel ID)
- PROPERTYCITY (city where the property is located)
- CLASSDESC (building class (description))
- GRADE (dwelling grade)
- GRADEDESC (dwelling grade description)
- OWNERDESC (owner description)
- SALEDATE (sale date)
- SALEPRICE (sale price)
- COUNTYBUILDING (county assessed value for building)
- COUNTYLAND (county assessed value for land)
- COUNTYTOTAL (county assessed value)
- FAIRMARKETBUILDING (county assessed fair market building value)
- FAIRMARKETLAND (county assessed fair market land value)
- FAIRMARKETTOTAL (county assessed fair market total value)
- TAXYEAR (current certified tax year)

## Step 2: Clean Up Data
Subset the assessments dataframe. Join the parcel boundaries to the assessments by parcel ID.

In [ ]:
# subset assessments to relevant columns
assessments_clean_df = assessments_df[[ 
    "PARID",
    "PROPERTYCITY",
    "CLASS",
    "CLASSDESC",
    "OWNERDESC",
    "GRADE",
    "SALEDATE",
    "SALEPRICE",
    "COUNTYBUILDING",
    "COUNTYLAND",
    "COUNTYTOTAL",
    "FAIRMARKETBUILDING",
    "FAIRMARKETLAND",
    "FAIRMARKETTOTAL",
    "TAXYEAR"
]].rename(columns=str.lower).copy()

# subset municipal boundaries to relevant columns
municipal_boundaries_clean_gdf = municipal_boundaries_gdf[[
    "NAME",
    "TYPE",
    "REGION",
    "SQMI",
    "EOC",
    "geometry"
]].rename(columns=str.lower).copy()

# separate saledate column into day, month, and year
assessments_clean_df[["sale_day", "sale_month", "sale_year"]] = assessments_clean_df["saledate"].str.split("-", expand=True)

# convert day, month, and year columns to integers
def change_to_int(df):
    return df.apply(pd.to_numeric, errors="coerce").astype("Int64")

assessments_clean_df[["sale_day", "sale_month", "sale_year"]] = change_to_int(
    assessments_clean_df[["sale_day", "sale_month", "sale_year"]]
)

# rename pin to parid in parcels dataframe
parcel_boundaries_clean = parcel_boundaries_df.rename(columns={"pin":"parid", "wkt":"geometry"}).copy()

# join assessments and parcel boundaries on parid
assessment_parcel_df = assessments_clean_df.merge(parcel_boundaries_clean, on="parid", how="left")

# set assessment parcel dataframe as a geopandas dataframe
assessment_parcel_df = gpd.GeoDataFrame(
    assessment_parcel_df,
    geometry=gpd.GeoSeries.from_wkt(assessment_parcel_df["geometry"]),
    crs="EPSG:2272"  # PA South State Plane (US feet)
)

# set municipal_boundaries_clean_gdf coordinate system to match assessment_parcel_df coordinate system
municipal_boundaries_clean_gdf = municipal_boundaries_clean_gdf.to_crs(assessment_parcel_df.crs)

assessment_parcel_df["centroid"] = assessment_parcel_df.geometry.centroid
assessment_parcel_df = assessment_parcel_df.rename(columns={"geometry":"polygeom"})
assessment_parcel_df = assessment_parcel_df.set_geometry("centroid")

# apply spatial join of assessment parcel dataframe with municipal boundaries
assessment_parcel_municipal_df = gpd.sjoin(
    assessment_parcel_df,
    municipal_boundaries_clean_gdf,
    how="inner",
    predicate="intersects"
)

# reset geometry back to polygeom
assessment_parcel_municipal_df = assessment_parcel_municipal_df.set_geometry("polygeom")

There should be 585,005 rows in the dataframe after the spatial join. If there is a discrepancy the join is wrong.

In [ ]:
# export assessment_parcel_municipal_df
parcel_df_analytical=(
    # convert centroid geometry to lat/lon columns
    assessment_parcel_municipal_df
    .assign(
        centroid_4326=lambda x: x["centroid"].to_crs("EPSG:4326"),
        lat=lambda x: x["centroid_4326"].y.astype("string"),
        lon=lambda x: x["centroid_4326"].x.astype("string")
    )
    .drop(columns = ["centroid", "centroid_4326"])
)

## Step 3: Save the data as a geojson

In [ ]:
parcel_df_analytical.to_file("assessor_parcels_analytic.geojson", driver="GeoJSON")